In [ ]:
import pandas as pd

initial_df = pd.read_csv("btcusdt_1h.csv")

final_df = initial_df
initial_df

FileNotFoundError: ignored

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
pip install ta

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# import pandas_ta as ta
import math
from ta.trend import MACD

# Assuming you have initial_df already defined with a 'Close' column
# Calculate the RSI indicator for initial_df
macd = MACD(close=initial_df['close'],window_slow = 49 , window_fast = 24 , window_sign = 9, fillna=False)

# Access the RSI values
macd_values = macd.macd_diff()

final_df['macd_vals'] = macd_values
final_df['macd_sig'] = 0
for i in range(len(macd_values)):
  if( np.isnan(macd_values[i])):
    final_df['macd_sig'][i] = 0
  elif(macd_values[i] > 0 ):
    final_df['macd_sig'][i] = 1
  else:
    final_df['macd_sig'][i] = -1



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# import pandas_ta as ta
import math
from ta.trend import AroonIndicator

# Assuming you have initial_df already defined with a 'Close' column
# Calculate the RSI indicator for initial_df
aroon = AroonIndicator( high = final_df['high'] , low = final_df['low'] , window = 12 , fillna = False)

# Access the RSI values
aroon_vals = aroon.aroon_indicator()

final_df['aroon_vals'] = aroon_vals
final_df['aroon_sig'] = 0
for i in range(len(aroon_vals)):
  if( np.isnan(aroon_vals[i])):
    final_df['aroon_sig'][i] = 0
  elif(aroon_vals[i] > 0 ):
    final_df['aroon_sig'][i] = 1
  else:
    final_df['aroon_sig'][i] = -1
print(final_df)


# Plot the RSI indicator
plt.figure(figsize=(12, 6))
plt.plot(final_df.index, final_df['aroon_vals'], label='Aroon Values', color='blue')

plt.xlabel('Date')
plt.ylabel('Value')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
#bollinger bands
from ta.utils import dropna
from ta.volume import OnBalanceVolumeIndicator

# Initialize On Balance Volume Indicator
indicator_obv = OnBalanceVolumeIndicator(close=final_df["close"],volume=final_df["volume"])
curr_sig=0
closed=True
final_df['obv_values'] = indicator_obv.on_balance_volume()
threshold = 100000 # Value threshold

signals_obv=[]
window_size=100
final_df["obv_ema"] = final_df['obv_values'].ewm(span=200, adjust=False).mean()
for i in range(len(final_df)):
    if final_df.iloc[i].isna().any():
        signals_obv.append(0)  # Not enough data for the rolling window

    else:
        '''
        window_start = i - (window_size // 2)  # 50 hours before
        window_end = i + (window_size // 2)  # 50 hours after
        window_values = final_df['obv_values'].iloc[window_start:window_end]
        diff = window_values.iloc[-1] - window_values.iloc[0]
        '''

        long_entry=final_df['obv_values'].iloc[i]>final_df['obv_ema'].iloc[i]
        long_exit=final_df['obv_values'].iloc[i]<final_df['obv_ema'].iloc[i]
        short_entry=long_exit
        short_exit=long_entry
        if long_entry or short_exit:
            signals_obv.append(1)
            '''
            if long_entry and closed==True:
                #previous trade closed, can open new one
                signals_obv.append(1)
                closed=False
                curr_sig=1
            else:
                if short_exit and curr_sig==-1:
                    closed=True
                    curr_sig=0
                    signals_obv.append(1)
                else:
                    signals_obv.append(0)'''
          # set -1
        elif short_entry or long_exit:
            signals_obv.append(-1)
            '''
            if short_entry and closed==True:
                signals_obv.append(-1)
                closed=False
                curr_sig=-1
            else:
                if long_exit and curr_sig==1:
                    closed=True
                    curr_sig=0
                    signals_obv.append(-1)
                else:
                    signals_obv.append(0)'''
        else:
            signals_obv.append(0)
final_df["obv_sig"]=signals_obv
final_signals_obv=pd.DataFrame(signals_obv)
#final_signals_obv.to_csv("obv_outputtttt.csv")

In [ ]:
final_df.head()

In [ ]:
import pandas_ta as ta
def ichimoku(data):
  ichi = ta.ichimoku(data['high'], data['low'], data['close'])
  data = pd.concat([data, ichi[0]], axis=1)
  return data
def ichi_long_entry(data,bar):
  if  data["close"].iloc[bar]>max(data["ISA_9"].iloc[bar], data["ISB_26"].iloc[bar]):
    return True
  else:
    return False

def ichi_long_exit(data,bar):
  if (data["ITS_9"].iloc[bar]<data["IKS_26"].iloc[bar] and data["ITS_9"].iloc[bar-1]>=data["IKS_26"].iloc[bar-1]):
    return True
  else:
    return False

def ichi_short_entry(data,bar):
  if  data["close"].iloc[bar]<min(data["ISA_9"].iloc[bar], data["ISB_26"].iloc[bar]) :
    return True
  else:
    return False

def ichi_short_exit(data,bar):
  if (data["ITS_9"].iloc[bar]>data["IKS_26"].iloc[bar] and data["ITS_9"].iloc[bar-1]<=data["IKS_26"].iloc[bar-1]):
    return True
  else:
    return False




In [ ]:
final_df=ichimoku(final_df)[:35208]
print((final_df.columns))

In [ ]:
#ichimoku signals
signals_ichi=[0]


curr_sig=0
closed=True
    #### set conditions for long_entry and long_exit
for i in range(1,len(final_df)):
    if final_df.iloc[i].isna().any():
        signals_ichi.append(0)

    else:
        long_entry=ichi_long_entry(final_df,i)
        long_exit=ichi_long_exit(final_df,i)
        short_entry=ichi_short_entry(final_df,i)
        short_exit=ichi_short_exit(final_df,i)
        if long_entry or short_exit:
              #go long
              signals_ichi.append(1)
              '''

              if long_entry and closed==True:
                  #previous trade closed, can open new one
                  signals_ichi.append(1)
                  closed=False
                  curr_sig=1
              else:
                  if short_exit and curr_sig==-1:
                      closed=True
                      curr_sig=0
                      signals_ichi.append(1)
                  else:
                      signals_ichi.append(0)'''
          # set -1
        elif short_entry or long_exit:
              signals_ichi.append(-1)
              '''

              if short_entry and closed==True:
                  signals_ichi.append(-1)
                  #print("no")
                  closed=False
                  curr_sig=-1
              else:
                  if long_exit and curr_sig==1:
                      closed=True
                      curr_sig=0
                      signals_ichi.append(-1)
                      #print("no")
                  else:
                      signals_ichi.append(0)'''
        else:
            signals_ichi.append(0)
final_df["ichimoku_sig"]=signals_ichi






In [ ]:
data_ichi=pd.DataFrame(signals_ichi)
data_ichi["signals"]=signals_ichi
data_ichi.to_csv("ichimokusignals.csv")

In [ ]:
#bollinger bands
from ta.utils import dropna
from ta.volatility import BollingerBands

# Initialize Bollinger Bands Indicator
indicator_bb = BollingerBands(close=final_df["close"], window=20, window_dev=2)
curr_sig=0
closed=True
final_df['bb_bbm_vals'] = indicator_bb.bollinger_mavg()
final_df['bb_bbh_vals'] = indicator_bb.bollinger_hband()
final_df['bb_bbl_vals'] = indicator_bb.bollinger_lband()
final_df['bb_bbhi'] = indicator_bb.bollinger_hband_indicator()
# Add Bollinger Band low indicator
final_df['bb_bbli'] = indicator_bb.bollinger_lband_indicator()
signals_bb=[]
for i in range(len(final_df)):
    if final_df.iloc[i].isna().any():
        signals_bb.append(0)

    else:
        long_entry=final_df['bb_bbhi'].iloc[i]
        long_exit=final_df["bb_bbli"].iloc[i]
        short_entry=long_exit
        short_exit=long_entry
        if long_entry or short_exit:
              signals_bb.append(1)
              '''
              #go long
              if long_entry and closed==True:
                  #previous trade closed, can open new one
                  signals_bb.append(1)
                  closed=False
                  curr_sig=1
              else:
                  if short_exit and curr_sig==-1:
                      closed=True
                      curr_sig=0
                      signals_bb.append(1)
                  else:
                      signals_bb.append(0)'''
          # set -1
        elif short_entry or long_exit:
              signals_bb.append(-1)
              '''
              if short_entry and closed==True:
                  signals_bb.append(-1)
                  closed=False
                  curr_sig=-1
              else:
                  if long_exit and curr_sig==1:
                      closed=True
                      curr_sig=0
                      signals_bb.append(-1)
                  else:
                      signals_bb.append(0)'''
        else:
            signals_bb.append(0)
final_df["bb_sig"]=signals_bb
final_signals_bb=pd.DataFrame(signals_bb)
final_signals_bb.to_csv("bb_outputtttt.csv")





In [ ]:
import pandas as pd
import ta

tester = ta.trend.ADXIndicator(final_df['high'], final_df['low'], final_df['close'])
final_df['adx'] = tester.adx() #Average Directional Index
final_df['pdi'] = tester.adx_pos() # + Directional Indicator
final_df['ndi'] = tester.adx_neg() # - Directional Indicator
adx_signals=[]
curr_sig=0
closed=True
#### set conditions for long_entry and long_exit
for bar in range(len(final_df)):
    if final_df.iloc[bar].isna().any():
        adx_signals.append(0)

    else:
        long_entry = final_df['pdi'][bar] > final_df['ndi'][bar]
        long_exit = final_df['pdi'][bar] < final_df['ndi'][bar]
        short_entry = final_df['pdi'][bar] < final_df['ndi'][bar]
        short_exit = final_df['pdi'][bar] > final_df['ndi'][bar]

        if long_entry or short_exit:
            adx_signals.append(1)
            '''
            if long_entry and closed==True:
                #previous trade closed, can open new one
                adx_signals.append(1)
                closed=False
                curr_sig=1
            else:
                if short_exit and curr_sig==-1:
                    closed=True
                    curr_sig=0
                    adx_signals.append(1)
                else:
                    adx_signals.append(0)'''
        elif short_entry or long_exit:
            adx_signals.append(-1)
            '''
            if short_entry and closed==True:
                adx_signals.append(-1)
                closed=False
                curr_sig=-1
            else:
                if long_exit and curr_sig==1:
                    closed=True
                    curr_sig=0
                    adx_signals.append(-1)
                else:
                    adx_signals.append(0)'''
        else:
            adx_signals.append(0)
final_df['adx_sig'] = adx_signals

In [ ]:
def rsi_long_entry(rsi_value):
    if (rsi_value <= 25 and rsi_value > 12.5) or rsi_value > 87.5:
        return True
    return False

def rsi_long_exit(rsi_value):
    if (rsi_value <= 87.5 and rsi_value > 75) or (rsi_value <= 12.5) or (rsi_value >= 25 and rsi_value < 75):
        return True
    return False

def rsi_short_entry(rsi_value):
    if (rsi_value <= 87.5 and rsi_value > 75) or rsi_value <= 12.5:
        return True
    return False

def rsi_short_exit(rsi_value):
    if (rsi_value <= 25 and rsi_value > 12.5) or (rsi_value > 87.5) or (rsi_value >= 25 and rsi_value < 75):
        return True
    return False


In [ ]:
import ta

tester = ta.momentum.RSIIndicator(close=final_df['close'], window=14, fillna=False)
rsi_values = tester.rsi()
final_df['rsi_values'] = rsi_values
rsi_signals=[]
curr_sig=0
closed=True
#### set conditions for long_entry and long_exit
for bar in range(len(final_df)):
    if final_df.iloc[bar].isna().any():
        rsi_signals.append(0)

    else:
        long_entry = rsi_long_entry(rsi_values[bar])
        long_exit = rsi_long_exit(rsi_values[bar])
        short_entry = rsi_short_entry(rsi_values[bar])
        short_exit = rsi_short_exit(rsi_values[bar])

        if long_entry or short_exit:
            rsi_signals.append(1)
            '''
            if long_entry and closed==True:
                #previous trade closed, can open new one
                rsi_signals.append(1)
                closed=False
                curr_sig=1
            else:
                if short_exit and curr_sig==-1:
                    closed=True
                    curr_sig=0
                    rsi_signals.append(1)
                else:
                    rsi_signals.append(0)'''
        # set -1
        elif short_entry or long_exit:
            rsi_signals.append(-1)
            '''
            if short_entry and closed==True:
                rsi_signals.append(-1)
                closed=False
                curr_sig=-1
            else:
                if long_exit and curr_sig==1:
                    closed=True
                    curr_sig=0
                    rsi_signals.append(-1)
                else:
                    rsi_signals.append(0)'''
        else:
            rsi_signals.append(0)
final_df['rsi_sig'] = rsi_signals

In [ ]:
#ichimoku signals
signals_final=[]
temp=[]
curr_sig=0
closed=True
import statistics
from statistics import mode
    #### set conditions for long_entry and long_exit
print(len(final_df))
for i in range(len(final_df)):
    if final_df.iloc[i].isna().any():
        signals_final.append(0)

    else:
        temp=[final_df["rsi_sig"].iloc[i],final_df["adx_sig"].iloc[i],final_df["macd_sig"].iloc[i],final_df["bb_sig"].iloc[i],final_df["obv_sig"].iloc[i],final_df["ichimoku_sig"].iloc[i],final_df["aroon_sig"].iloc[i]]
        dic={}
        long_entry=0
        short_exit=0
        short_entry=0
        long_exit=0
        for j in [-1,0,1]:
            dic[j]=0
        for j in temp:
            dic[j]+=1
        if dic[-1]>=3:
          short_entry=True
          long_exit=True
          long_entry=False
          short_exit=False
        elif dic[1]>=3:
          long_entry=True
          long_exit=False
          short_entry=False
          short_exit=True
        else:
          long_entry=False
          long_exit=False
          short_entry=False
          short_exit=False

        if long_entry or short_exit:
              #go long
              if long_entry and closed==True:
                  #previous trade closed, can open new one
                  signals_final.append(1)
                  closed=False
                  curr_sig=1
              else:
                  if short_exit and curr_sig==-1:
                      closed=True
                      curr_sig=0
                      signals_final.append(1)
                  else:
                      signals_final.append(0)
          # set -1
        elif short_entry or long_exit:
              if short_entry and closed==True:
                  signals_final.append(-1)
                  closed=False
                  curr_sig=-1
              else:
                  if long_exit and curr_sig==1:
                      closed=True
                      curr_sig=0
                      signals_final.append(-1)
                  else:
                      signals_final.append(0)
        else:
            signals_final.append(0)
final_df["final_sig"]=signals_final






In [ ]:
final_df

In [ ]:
# final_signals_values=pd.DataFrame(signals_final)
# final_signals_values.to_csv("outputtttt.csv")